<a href="https://colab.research.google.com/github/gopika-vit/Projects-AI/blob/main/Nutrition_and_Diet_Plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import random
import json
import os
from datetime import datetime




# ------------------------------
# Utility Functions
# ------------------------------

def load_users():
    """Load saved user profiles from JSON file."""
    if os.path.exists("users.json"):
        with open("users.json", "r") as f:
            return json.load(f)
    return {}

def save_users(users):
    """Save user profiles to JSON file."""
    with open("users.json", "w") as f:
        json.dump(users, f, indent=4)

def calculate_bmr(gender, weight, height, age):
    if gender.lower() == 'male':
        return 88.362 + (13.397 * weight) + (4.799 * height) - (5.677 * age)
    elif gender.lower() == 'female':
        return 447.593 + (9.247 * weight) + (3.098 * height) - (4.330 * age)
    else:
        return (88.362 + 447.593)/2 + ((13.397 + 9.247)/2 * weight) + ((4.799 + 3.098)/2 * height) - ((5.677 + 4.330)/2 * age)

def get_activity_multiplier(activity_level):
    levels = {
        'sedentary': 1.2,
        'lightly active': 1.375,
        'moderately active': 1.55,
        'very active': 1.725,
        'super active': 1.9
    }
    return levels.get(activity_level.lower(), 1.2)

def generate_meal_plan(answers, diet_type, goal):
    fruits = answers['49'].split(',') if answers['49'] else ['fruit']
    vegetables = answers['50'].split(',') if answers['50'] else ['veggies']
    proteins = answers['51'].split(',') if answers['51'] else ['tofu']
    carbs = answers['52'].split(',') if answers['52'] else ['rice']
    dislikes = answers['53'].split(',') if answers['53'] else []

    # Filter out disliked foods
    fruits = [f for f in fruits if f not in dislikes]
    vegetables = [v for v in vegetables if v not in dislikes]
    proteins = [p for p in proteins if p not in dislikes]
    carbs = [c for c in carbs if c not in dislikes]

    # Fallbacks if lists are empty after filtering
    fruits = fruits if fruits else ['fruit']
    vegetables = vegetables if vegetables else ['veggies']
    proteins = proteins if proteins else ['tofu']
    carbs = carbs if carbs else ['rice']

    # Adjust for dietary preference
    if 'vegan' in diet_type:
        proteins = [p for p in proteins if p not in ['paneer', 'cheese', 'yogurt', 'milk']]
        proteins = proteins if proteins else ['tofu']

    # Select random items for variety, ensuring at least one item
    breakfast_fruit = random.choice(fruits)
    lunch_protein = random.choice(proteins)
    lunch_veggie = random.choice(vegetables)
    lunch_carb = random.choice(carbs)
    dinner_protein = random.choice([p for p in proteins if p != lunch_protein] if len(proteins) > 1 else proteins)
    dinner_veggie = random.choice([v for v in vegetables if v != lunch_veggie] if len(vegetables) > 1 else vegetables)
    dinner_carb = random.choice([c for c in carbs if c != lunch_carb] if len(carbs) > 1 else carbs)
    snack_fruit = random.choice([f for f in fruits if f != breakfast_fruit] if len(fruits) > 1 else fruits)

    # Adjust for goal (e.g., weight gain: emphasize calorie-dense foods)
    portion_note = " (increase portion sizes or add nuts/seeds for extra calories)" if 'gain' in goal else ""

    meal_plan = []
    if 'vegetarian' in diet_type or 'vegan' in diet_type:
        meal_plan.append(f"Breakfast: Oatmeal with {breakfast_fruit} and {'yogurt' if 'vegetarian' in diet_type else 'plant-based milk'}.")
        meal_plan.append(f"Lunch: {lunch_protein} salad with {lunch_veggie} and {lunch_carb}{portion_note}.")
        meal_plan.append(f"Dinner: {dinner_protein} curry with {dinner_veggie} and {dinner_carb}{portion_note}.")
        meal_plan.append(f"Snacks: {snack_fruit}, {'yogurt' if 'vegetarian' in diet_type else 'seeds'}.")
    else:
        meal_plan.append(f"Breakfast: Oatmeal with {breakfast_fruit} and nuts (avoid if allergic).")
        meal_plan.append(f"Lunch: Grilled {lunch_protein} with {lunch_veggie} and {lunch_carb}{portion_note}.")
        meal_plan.append(f"Dinner: {dinner_protein} with {dinner_veggie} and {dinner_carb}{portion_note}.")
        meal_plan.append(f"Snacks: {snack_fruit}, yogurt, or nuts.")

    return meal_plan

def main():
    users = load_users()
    print("\n🍎 Welcome to the Nutrition and Diet Recommendation Expert System")
    print("This expert system will ask personalized questions to generate your plan.\n")

    name = input("Enter your name: ").strip().title()

    # Check if user profile exists
    user_data = users.get(name, {})

    if user_data:
        print(f"\nWelcome back, {name}! (Last update: {user_data.get('last_updated', 'N/A')})")
        reuse = input("Would you like to reuse your saved data? (yes/no): ").lower()
        if reuse != "no":
            print("✅ Reusing your previous data. You can update specific answers below.\n")
        else:
            user_data = {}
    else:
        print(f"\nHello {name}! Let's set up your personalized profile.\n")

    # Helper for adaptive questioning
    def ask(prompt, key, cast=str):
        """Ask question and reuse stored value if present."""
        if key in user_data:
            val = input(f"{prompt} (current: {user_data[key]}) — press Enter to keep: ").strip()
            if val == "":
                return user_data[key]
        else:
            val = input(prompt + " ").strip()
        try:
            return cast(val)
        except ValueError:
            return val

    # ------------------------------

    print("Please answer the following 60 questions honestly for accurate recommendations.")
    print("For yes/no questions, enter 'yes' or 'no'.")
    print("For numerical questions, enter numbers only.")
    print("For list or open-ended questions, enter comma-separated values or descriptions.")

    questions = [
        "1. What is your name?",
        "2. What is your age? (in years)",
        "3. What is your gender? (male/female/other)",
        "4. What is your height? (in cm)",
        "5. What is your weight? (in kg)",
        "6. Do you have diabetes? (yes/no)",
        "7. Do you have hypertension? (yes/no)",
        "8. Do you have high cholesterol? (yes/no)",
        "9. Do you have thyroid issues? (yes/no)",
        "10. Do you have heart disease? (yes/no)",
        "11. Do you have kidney problems? (yes/no)",
        "12. Do you have liver problems? (yes/no)",
        "13. Do you have anemia? (yes/no)",
        "14. Do you have osteoporosis? (yes/no)",
        "15. Do you have any other medical conditions? (yes/no or describe)",
        "16. Are you on any medications? (yes/no)",
        "17. If on medications, what are they? (comma-separated, or 'none')",
        "18. Are you allergic to nuts? (yes/no)",
        "19. Are you allergic to dairy? (yes/no)",
        "20. Are you allergic to gluten? (yes/no)",
        "21. Are you allergic to shellfish? (yes/no)",
        "22. Are you allergic to eggs? (yes/no)",
        "23. Are you allergic to soy? (yes/no)",
        "24. Are you allergic to peanuts? (yes/no)",
        "25. Are you allergic to wheat? (yes/no)",
        "26. Are you allergic to fish? (yes/no)",
        "27. Do you have any other allergies? (yes/no or describe)",
        "28. What is your dietary preference? (vegetarian/vegan/pescatarian/omnivore/other)",
        "29. What is your activity level? (sedentary/lightly active/moderately active/very active/super active)",
        "30. How many hours do you sleep per night on average?",
        "31. What is your stress level? (low/medium/high)",
        "32. Do you smoke? (yes/no)",
        "33. How often do you drink alcohol? (never/occasionally/regularly)",
        "34. What is your daily water intake? (in liters)",
        "35. Do you take supplements? (yes/no)",
        "36. Do you take multivitamins? (yes/no)",
        "37. Do you take vitamin D supplements? (yes/no)",
        "38. Do you take calcium supplements? (yes/no)",
        "39. Do you take omega-3 supplements? (yes/no)",
        "40. Do you take protein supplements? (yes/no)",
        "41. What is your primary goal? (weight loss/weight gain/maintenance/muscle building/improve health/other)",
        "42. If weight loss or gain, how much is the targeted weight in kg?",
        "43. What is your timeline for your goal? (in months)",
        "44. How many meals do you eat per day?",
        "45. Do you eat breakfast regularly? (yes/no)",
        "46. Do you eat lunch regularly? (yes/no)",
        "47. Do you eat dinner regularly? (yes/no)",
        "48. How many snacks do you have per day?",
        "49. What are your favorite fruits? (comma-separated)",
        "50. What are your favorite vegetables? (comma-separated)",
        "51. What are your preferred protein sources? (comma-separated)",
        "52. What are your preferred carbohydrate sources? (comma-separated)",
        "53. What foods do you dislike? (comma-separated)",
        "54. Do you have any specific dietary restrictions? (e.g., low carb/keto/paleo/none)",
        "55. How many days per week do you exercise?",
        "56. What type of exercise do you do? (cardio/strength/yoga/mixed/none)",
        "57. What is the average duration of your exercise sessions? (in minutes)",
        "58. Do you have any injuries or physical limitations? (yes/no or describe)",
        "59. What is your motivation level for changing your diet? (low/medium/high)",
        "60. Any additional comments or preferences for your diet plan?"
    ]

    answers = {}
    for q in questions:
      key = q.split('.')[0].strip()
      old_val = user_data.get(key, "")
      prompt = f"{q} (current: {old_val}) — press Enter to keep: " if old_val else f"{q} "
      ans = input(prompt).strip()
      answers[key] = ans if ans else old_val

    try:
        age = float(answers['2'])
        height = float(answers['4'])
        weight = float(answers['5'])
        gender = answers['3']
        activity_level = answers['29']
        goal = answers['41'].lower()
        water_intake = float(answers['34']) if answers['34'] else 0
        sleep_hours = float(answers['30']) if answers['30'] else 0
        exercise_days = float(answers['55']) if answers['55'] else 0
        target_weight_change = float(answers['42']) - weight if answers['42'] and 'gain' in goal else float(answers['42']) if answers['42'] else 0
        timeline_months = float(answers['43']) if answers['43'] else 0
    except ValueError:
        print("Error: Invalid input for numerical fields. Using defaults where possible.")
        age = 30
        height = 170
        weight = 70
        water_intake = 2
        sleep_hours = 7
        exercise_days = 3
        target_weight_change = 0
        timeline_months = 3

    bmi = weight / ((height / 100) ** 2)
    bmr = calculate_bmr(gender, weight, height, age)
    activity_multiplier = get_activity_multiplier(activity_level)
    tdee = bmr * activity_multiplier
    daily_calories = tdee
    if 'loss' in goal:
        daily_calories -= 500
    elif 'gain' in goal:
        daily_calories += 500
    elif 'muscle' in goal:
        daily_calories += 300
    carbs = (daily_calories * 0.4) / 4
    protein = (daily_calories * 0.3) / 4
    fat = (daily_calories * 0.3) / 9

    recommendations = []
    if answers['6'].lower() == 'yes':
        recommendations.append("Focus on low glycemic index foods to manage blood sugar.")
    if answers['7'].lower() == 'yes':
        recommendations.append("Reduce sodium intake and increase potassium-rich foods.")
    if answers['8'].lower() == 'yes':
        recommendations.append("Incorporate more soluble fiber and healthy fats.")
    if answers['32'].lower() == 'yes':
        recommendations.append("Consider quitting smoking for better nutrient absorption.")
    if sleep_hours < 7:
        recommendations.append("Aim for at least 7-8 hours of sleep for optimal health.")
    if water_intake < 2:
        recommendations.append("Increase water intake to at least 2-3 liters per day.")
    if exercise_days < 3:
        recommendations.append("Try to exercise at least 3-5 days per week for better health and weight gain support.")
    if answers['31'].lower() == 'high':
        recommendations.append("Incorporate stress-reducing foods like whole grains, nuts, and magnesium-rich foods (e.g., spinach).")

    diet_type = answers['28'].lower()
    if 'vegan' in diet_type:
        recommendations.append("Ensure adequate plant-based protein sources like beans, lentils, tofu.")
    elif 'vegetarian' in diet_type:
        recommendations.append("Include dairy and plant-based proteins like paneer, soya, and lentils for balanced nutrition.")

    allergies = []
    for q_num in ['18', '19', '20', '21', '22', '23', '24', '25', '26']:
        if answers[q_num].lower() == 'yes':
            allergy_map = {
                '18': 'nuts', '19': 'dairy', '20': 'gluten', '21': 'shellfish',
                '22': 'eggs', '23': 'soy', '24': 'peanuts', '25': 'wheat', '26': 'fish'
            }
            allergies.append(allergy_map[q_num])
    if answers['27'].lower() != 'no':
        allergies.append(answers['27'].replace('yes ', ''))
    if allergies:
        recommendations.append(f"Avoid foods associated with: {', '.join(allergies)}.")

    print("\n🎯🥗--- Personalized Nutrition and Diet Recommendations ---🥗🎯")
    print(f"Hello, {answers['1']}! Based on your answers:")
    print(f"📊Your BMI is {bmi:.2f} {'(underweight, supporting weight gain goal)' if bmi < 18.5 else ''}.")
    print(f"Your estimated BMR is {bmr:.2f} calories/day.")
    print(f"Your estimated TDEE (daily calorie needs for maintenance) is {tdee:.2f} calories/day.")
    print(f"Recommended daily calories for your goal: {daily_calories:.2f}")
    print(f"Suggested macros: Carbs ~{carbs:.0f}g, Protein ~{protein:.0f}g, Fat ~{fat:.0f}g")

    if target_weight_change and timeline_months:
        weekly_change = target_weight_change / (timeline_months * 4)
        if 'gain' in goal:
            print(f"To achieve {target_weight_change:.1f}kg weight gain (from {weight}kg to {answers['42']}kg) in {timeline_months:.1f} months, aim for ~{weekly_change:.2f}kg per week.")
        else:
            print(f"To achieve {target_weight_change:.1f}kg change in {timeline_months:.1f} months, aim for ~{weekly_change:.2f}kg per week.")

    if recommendations:
        print("\nSpecific Recommendations:")
        for rec in recommendations:
            print(f"- {rec}")

    print("\n🍎 Sample Daily Meal Plan (Customized based on your preferences):")
    for meal in generate_meal_plan(answers, diet_type, goal):
        print(meal)
    print(f"Incorporate your favorites: Fruits - {answers['49'] if answers['49'] else 'none provided'}, Veggies - {answers['50'] if answers['50'] else 'none provided'}.")
    print(f"Avoid dishes like: {answers['53'] if answers['53'] else 'none provided'}.")
    # ------------------------------
    # Save updated data
    # ------------------------------
    user_data.update(answers)
    user_data['last_updated'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    users[name] = user_data
    save_users(users)
    print(f"\n💾 Your profile for {name} has been saved successfully.")
    print("\nConsult a doctor or nutritionist for personalized advice. This is a basic expert system.")



if __name__ == "__main__":
    main()




🍎 Welcome to the Nutrition and Diet Recommendation Expert System
This expert system will ask personalized questions to generate your plan.

Enter your name: Anjana

Hello Anjana! Let's set up your personalized profile.

Please answer the following 60 questions honestly for accurate recommendations.
For yes/no questions, enter 'yes' or 'no'.
For numerical questions, enter numbers only.
For list or open-ended questions, enter comma-separated values or descriptions.
1. What is your name? Anjana
2. What is your age? (in years) 26
3. What is your gender? (male/female/other) female
4. What is your height? (in cm) 160
5. What is your weight? (in kg) 64
6. Do you have diabetes? (yes/no) no
7. Do you have hypertension? (yes/no) no
8. Do you have high cholesterol? (yes/no) no
9. Do you have thyroid issues? (yes/no) no
10. Do you have heart disease? (yes/no) no
11. Do you have kidney problems? (yes/no) no
12. Do you have liver problems? (yes/no) no
13. Do you have anemia? (yes/no) no
14. Do you 

In [ ]:
import json

# --- Knowledge Base ---

# Meal calorie distribution (percentages)
meal_distribution = {
    "breakfast": 0.3,
    "lunch": 0.4,
    "dinner": 0.3
}

# Nutrient-rich food sources (for deficiencies)
nutrient_sources = {
    "vitamin d": ["salmon", "egg yolks", "fortified milk", "mushrooms"],
    "iron": ["spinach", "lentils", "red meat", "pumpkin seeds"],
    "calcium": ["milk", "yogurt", "tofu", "broccoli"],
    "vitamin b12": ["eggs", "fish", "chicken", "fortified cereals"],
    "vitamin c": ["oranges", "kiwi", "bell peppers", "strawberries"]
}

# --- Food Calorie Knowledge Base ---
food_calories = {
    "rice (1 cup)": 200,
    "roti (1 piece)": 100,
    "dal (1 cup)": 180,
    "chicken curry (1 serving)": 250,
    "paneer curry (1 serving)": 220,
    "boiled egg (1)": 78,
    "milk (1 glass)": 120,
    "vegetable salad (1 bowl)": 90,
    "banana (1)": 105,
    "apple (1)": 95,
    "fried rice (1 plate)": 350,
    "idli (2 pieces)": 140,
    "dosa (1)": 170,
    "upma (1 bowl)": 250,
    "poha (1 bowl)": 230
}

# --- Core Functions ---

def estimate_meal_calories():
    """Let user input what they ate and estimate total calories"""
    total = 0
    print("\nEnter foods you ate (type 'done' when finished):")
    while True:dosa
        item = input("Food item: ").lower().strip()
        if item == 'done':
            break
        elif item in food_calories:
            total += food_calories[item]
            print(f"  → {item.title()} = {food_calories[item]} kcal added.")
        else:
            print("  ⚠ Item not found in database. Try a different name.")
    print(f"\nEstimated total meal calories: {total} kcal.")
    return total

def calculate_bmr(gender, weight, height, age):
    """Calculate BMR using Mifflin-St Jeor formula"""
    if gender.lower() == 'male':
        return 88.36 + (13.4 * weight) + (4.8 * height) - (5.7 * age)
    else:
        return 447.6 + (9.2 * weight) + (3.1 * height) - (4.3 * age)

def get_activity_multiplier(activity_level):
    """Return TDEE multiplier based on activity level"""
    levels = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725,
        "very active": 1.9
    }
    return levels.get(activity_level.lower(), 1.2)

def meal_wise_calories(total_calories):
    """Distribute daily calories into meals"""
    meal_plan = {}
    for meal, ratio in meal_distribution.items():
        meal_plan[meal] = round(total_calories * ratio)
    return meal_plan

def suggest_for_deficiency(deficiency):
    """Suggest foods for specific vitamin/mineral deficiency"""
    deficiency = deficiency.lower().strip()
    if deficiency in nutrient_sources:
        foods = nutrient_sources[deficiency]
        return f"For {deficiency.title()} deficiency, include: {', '.join(foods)}."
    else:
        return "No specific recommendations found for that deficiency."

# --- Main Program Flow ---

print("=== Nutrition and Diet Recommendation Expert System ===")

# ⿡ Collect user data
gender = input("Enter your gender (Male/Female): ")
age = int(input("Enter your age: "))
weight = float(input("Enter your weight (kg): "))
height = float(input("Enter your height (cm): "))
activity_level = input("Enter your activity level (sedentary/light/moderate/active/very active): ")
goal = input("Your goal (weight loss/maintenance/weight gain): ")

# ⿢ Calculate BMR & TDEE
bmr = calculate_bmr(gender, weight, height, age)
tdee = bmr * get_activity_multiplier(activity_level)

# ⿣ Adjust calories based on goal
if goal.lower() == "weight loss":
    total_calories = tdee - 500
elif goal.lower() == "weight gain":
    total_calories = tdee + 500
else:
    total_calories = tdee

print(f"\nYour recommended total calorie intake: {round(total_calories)} kcal per day.")

# ⿤ Meal-wise breakdown
meal_plan = meal_wise_calories(total_calories)
print("\nSuggested meal-wise calorie distribution:")
for meal, cals in meal_plan.items():
    print(f"  {meal.title()}: {cals} kcal")

# ⿥ Vitamin/mineral deficiency suggestions
deficiency = input("\nDo you have any vitamin/mineral deficiency? (e.g., Vitamin D, Iron, Calcium): ")
if deficiency.strip():
    print(suggest_for_deficiency(deficiency))
else:
    print("No specific deficiencies reported. Balanced diet recommended.")


# After printing meal-wise distribution
print("\n--- Optional: Track Your Actual Meal Intake ---")
choice = input("Do you want to calculate calories of your last meal? (yes/no): ")
if choice.lower() == 'yes':
    meal_calories = estimate_meal_calories()
    print("\n📊 Comparison with suggested meal goal:")
    for meal, cals in meal_plan.items():
        if abs(meal_calories - cals) < 100:
            print(f"✅ Your meal matches your {meal.title()} calorie goal (~{cals} kcal).")
            break
    else:
        if meal_calories > max(meal_plan.values()):
            print("⚠ Your meal exceeded recommended calories. Try lighter options next time.")
        else:
            print("ℹ Your meal was under the recommended calories.")

=== Nutrition and Diet Recommendation Expert System ===
Enter your gender (Male/Female): Female
Enter your age: 26
Enter your weight (kg): 60
Enter your height (cm): 152
Enter your activity level (sedentary/light/moderate/active/very active): active
Your goal (weight loss/maintenance/weight gain): weight loss

Your recommended total calorie intake: 1844 kcal per day.

Suggested meal-wise calorie distribution:
  Breakfast: 553 kcal
  Lunch: 738 kcal
  Dinner: 553 kcal

Do you have any vitamin/mineral deficiency? (e.g., Vitamin D, Iron, Calcium): calcium
For Calcium deficiency, include: milk, yogurt, tofu, broccoli.

--- Optional: Track Your Actual Meal Intake ---
Do you want to calculate calories of your last meal? (yes/no): yes

Enter foods you ate (type 'done' when finished):
Food item: biriyani
  ⚠ Item not found in database. Try a different name.
Food item: poha
  ⚠ Item not found in database. Try a different name.
Food item: rice
  ⚠ Item not found in database. Try a different name

KeyboardInterrupt: Interrupted by user